# Beyond Last-Click - Notebook 02
## Data Cleaning + SQL Loading 

**What this notebook does:**
1. Load both cleaned datasets
2. Check and fix data quality issues 
3. Connect to PostgreSQL database
4. Load both tables into the database
5. Run SQL queries to verify and explore

In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine, text
from dotenv import dotenv_values

print("✅ Libraries loaded")
print(f"Pandas version: {pd.__version__}")

✅ Libraries loaded
Pandas version: 2.3.3


## Part 1 - Load Datasets

In [2]:
# Load YouTube clean dataset
df_yt = pd.read_csv('../02_Datasets/processed/youtube_influencers_clean.csv')

# Load A/B Testing clean dataset
df_ab = pd.read_csv('../02_Datasets/processed/ab_testing_clean.csv')

print("=== YOUTUBE DATASET ===")
print(f"Shape: {df_yt.shape[0]:,} rows x {df_yt.shape[1]} columns")

print("\n=== A/B TESTING DATASET ===")
print(f"Shape: {df_ab.shape[0]:,} rows x {df_ab.shape[1]} columns")

=== YOUTUBE DATASET ===
Shape: 7,281 rows x 10 columns

=== A/B TESTING DATASET ===
Shape: 588,101 rows x 6 columns


## Part 2 - Data Quality Check

In [3]:
print("=== YOUTUBE — QUALITY CHECK ===")
print(f"\n1. Missing values:")
print(df_yt.isnull().sum())
print(f"\n2. Duplicate channel IDs: {df_yt['channel_id'].duplicated().sum()}")
print(f"\n3. YouTube creators with 0 subscribers: {(df_yt['subscribers'] == 0).sum()}")
print(f"\n4. Tier distribution:")
print(df_yt['tier'].value_counts())
print(f"\n5. Data types:")
print(df_yt.dtypes)

=== YOUTUBE — QUALITY CHECK ===

1. Missing values:
channel_name           0
channel_id             0
subscribers            0
total_views            0
video_count            0
avg_views_per_video    0
view_per_subscriber    0
country                0
tier                   0
niche                  0
dtype: int64

2. Duplicate channel IDs: 0

3. YouTube creators with 0 subscribers: 0

4. Tier distribution:
tier
nano     3907
macro    1339
mega     1070
micro     965
Name: count, dtype: int64

5. Data types:
channel_name            object
channel_id              object
subscribers              int64
total_views              int64
video_count              int64
avg_views_per_video      int64
view_per_subscriber    float64
country                 object
tier                    object
niche                   object
dtype: object


In [4]:
print("=== A/B TESTING — QUALITY CHECK ===")
print(f"\n1. Missing values:")
print(df_ab.isnull().sum())
print(f"\n2. Duplicate user IDs: {df_ab['user id'].duplicated().sum()}")
print(f"\n3. Test groups:")
print(df_ab['test group'].value_counts())
print(f"\n4. Data types:")
print(df_ab.dtypes)

=== A/B TESTING — QUALITY CHECK ===

1. Missing values:
user id          0
test group       0
converted        0
total ads        0
most ads day     0
most ads hour    0
dtype: int64

2. Duplicate user IDs: 0

3. Test groups:
test group
ad     564577
psa     23524
Name: count, dtype: int64

4. Data types:
user id           int64
test group       object
converted          bool
total ads         int64
most ads day     object
most ads hour     int64
dtype: object


## Part 3 - Clean Column Names 

In [5]:
# Fix column names - remove spaces, use underscores
df_ab.columns = df_ab.columns.str.strip().str.replace(' ', '_')

print("A/B Testing columns after fix:")
print(df_ab.columns.tolist())

print("\nYouTube columns (already clean):")
print(df_yt.columns.tolist())

A/B Testing columns after fix:
['user_id', 'test_group', 'converted', 'total_ads', 'most_ads_day', 'most_ads_hour']

YouTube columns (already clean):
['channel_name', 'channel_id', 'subscribers', 'total_views', 'video_count', 'avg_views_per_video', 'view_per_subscriber', 'country', 'tier', 'niche']


### ⚠️ Distribution Note — Nano Tier & Filter Decision

The raw dataset contains nano channels ranging from 1 to 9,999 subscribers.
Channels with very few subscribers (e.g. 1–10) produce extreme view_per_subscriber ratios
that are not representative of real influencer activity.

**Filter applied in analysis: subscribers ≥ 1,000**

This aligns with the industry-standard nano tier definition (1K–10K subscribers)
and ensures all channels in scope represent real, discoverable creators.

| Filter | Channels removed | Reason |
|--------|-----------------|--------|
| subscribers < 1,000 | ~3,433 | Below industry nano threshold — spam, ghost, or hobby accounts |

→ Final analysis dataset: **3,848 channels** (subscribers ≥ 1,000)
→ All SQL and statistical tests use MEDIAN as central tendency (right-skewed distribution)
→ Kruskal-Wallis is robust to this skew (rank-based, non-parametric)

## Part 4 - Connect to PostgreSQL

In [6]:
# Load credentials from .env
config = dotenv_values('../.env')

pg_user   = config['POSTGRES_USER']
pg_pass   = config['POSTGRES_PASS']
pg_host   = config['POSTGRES_HOST']
pg_port   = config['POSTGRES_PORT']
pg_db     = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']

# Build connection URL — same format as course
url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

# Create engine
engine = create_engine(url, echo=False)

print("✅ Engine created — connected to PostgreSQL!")
print(f"📂 Schema: {pg_schema}")

✅ Engine created — connected to PostgreSQL!
📂 Schema: s_soheilazamani


## Part 5 - Load Tables into Database

In [7]:
# Set schema, then write both tables to PostgreSQL
with engine.begin() as conn:
    conn.execute(text(f'SET search_path TO {pg_schema};'))

df_yt.to_sql('youtube_channels', engine, if_exists='replace', index=False)
df_ab.to_sql('ab_testing', engine, if_exists='replace', index=False)

print("✅ Tables written to PostgreSQL!")
print(f"   → youtube_channels: {len(df_yt):,} rows")
print(f"   → ab_testing: {len(df_ab):,} rows")
print(f"\n💡 Check DBeaver → s_soheilazamani → Tables to verify!")

✅ Tables written to PostgreSQL!
   → youtube_channels: 7,281 rows
   → ab_testing: 588,101 rows

💡 Check DBeaver → s_soheilazamani → Tables to verify!


## Part 6 - SQL Queries

### Query 1 - Preview both tables 

In [8]:
with engine.begin() as conn:
    conn.execute(text(f'SET search_path TO {pg_schema};'))

print("=== YOUTUBE TABLE - FIRST 3 ROWS ===")
result = pd.read_sql(sql=text('SELECT * FROM youtube_channels LIMIT 3;'), con=engine)
print(result)

print("\n=== A/B TESTING TABLE — first 3 rows ===")
result2 = pd.read_sql(sql=text('SELECT * FROM ab_testing LIMIT 3;'), con=engine)
print(result2)

=== YOUTUBE TABLE - FIRST 3 ROWS ===
       channel_name                channel_id  subscribers  total_views  \
0            UBELLA  UCQmtEz5330JeLEIaoInalgA         6880       176444   
1  PAINTEDBYSPENCER  UC0lj5odQ1wq2ZXgQKEZIx_Q       992000     88774510   
2  Risa Does Makeup  UCVtgrv6KHBNcRxDTys9TCcQ       476000     57754188   

   video_count  avg_views_per_video  view_per_subscriber country   tier  \
0           36                 4901               0.7124      ZA   nano   
1          188               472205               0.4760      US  macro   
2         1246                46352               0.0974      US  macro   

    niche  
0  Beauty  
1  Beauty  
2  Beauty  

=== A/B TESTING TABLE — first 3 rows ===
   user_id test_group  converted  total_ads most_ads_day  most_ads_hour
0  1069124         ad      False        130       Monday             20
1  1119715         ad      False         93      Tuesday             22
2  1144181         ad      False         21      Tuesda

### Query 2 - Engagement by tier (H1 preview)

In [9]:
query = text("""
SELECT 
    tier,
    COUNT(*) as channel_count,
    ROUND(AVG(subscribers)::numeric, 0) as avg_subscribers,
    ROUND(AVG(view_per_subscriber)::numeric, 4) as avg_engagement
FROM youtube_channels
GROUP BY tier
ORDER BY avg_engagement DESC
""")
result = pd.read_sql(sql=query, con=engine)
print("=== ENGAGEMENT BY TIER ===")
print(result)

=== ENGAGEMENT BY TIER ===
    tier  channel_count  avg_subscribers  avg_engagement
0   nano           3907            979.0         29.2168
1  micro            965          40579.0          2.7229
2  macro           1339         392763.0          0.9933
3   mega           1070        5763963.0          0.5944


### Query 3 - Conversion rate by group (H4 preview)

In [10]:
query2 = text("""
SELECT
    test_group,
    COUNT(*) as total_users,
    SUM(CASE WHEN converted = TRUE THEN 1 ELSE 0 END) as conversions,
    ROUND(AVG(CASE WHEN converted = TRUE THEN 1.0 ELSE 0.0 END) * 100, 2) as conversion_rate_pct
FROM ab_testing
GROUP BY test_group
""")
result2 = pd.read_sql(sql=query2, con=engine)
print("=== CONVERSION RATE BY GROUP ===")
print(result2)

=== CONVERSION RATE BY GROUP ===
  test_group  total_users  conversions  conversion_rate_pct
0         ad       564577        14423                 2.55
1        psa        23524          420                 1.79


### Query 4 - Top 10 most engaged niches

In [11]:
query3 = text("""
SELECT
    niche,
    COUNT(*) as creator_count,
    ROUND(AVG(view_per_subscriber)::numeric, 4) as avg_engagement,
    ROUND(AVG(subscribers)::numeric, 0) as avg_subscribers
FROM youtube_channels
GROUP BY niche
ORDER BY avg_engagement DESC
""")
result3 = pd.read_sql(sql=query3, con=engine)
print("=== ENGAGEMENT BY NICHE ===")
print(result3)

=== ENGAGEMENT BY NICHE ===
              niche  creator_count  avg_engagement  avg_subscribers
0           Fashion           1095         32.8163         289935.0
1          Shopping            452         24.6737         771516.0
2            Travel            593         20.8646         592185.0
3           Fitness            800         14.9670         975623.0
4            Beauty           1274         14.3375         686138.0
5  Film & Streaming            884         13.5055         708030.0
6            Gaming            554          9.1251        3545179.0
7         Lifestyle            893          8.1102         623708.0
8              Food            736          6.5334        1246504.0


## Part 7 - Close Database Connection 

In [12]:
engine.dispose()
print("✅ PostgreSQL engine disposed")
print("\n=== NOTEBOOK 02 COMPLETE ===")
print("What we did:")
print("  1. Loaded both cleaned datasets")
print("  2. Checked data quality — 0 issues found")
print("  3. Fixed column names")
print("  4. Connected to PostgreSQL (SPICED course database)")
print("  5. Loaded tables into schema: s_soheilazamani")
print("  6. Ran 4 SQL queries — findings confirmed")
print("\nNext: Notebook 03 — Statistical Hypothesis Testing")

✅ PostgreSQL engine disposed

=== NOTEBOOK 02 COMPLETE ===
What we did:
  1. Loaded both cleaned datasets
  2. Checked data quality — 0 issues found
  3. Fixed column names
  4. Connected to PostgreSQL (SPICED course database)
  5. Loaded tables into schema: s_soheilazamani
  6. Ran 4 SQL queries — findings confirmed

Next: Notebook 03 — Statistical Hypothesis Testing
